# Supplementary Figure 9 | Fine-mapping and annotation enrichment

**a.** Number of SuSiE fine-mapped eQTL credible sets (c.s.) per gene, pooled across all tissues and stratified by biotype (novel lncRNA, existing lncRNA, protein-coding).

**b.** Number of variants per eQTL c.s., pooled across tissues and stratified by biotype as in (a). Counts ≥250 are binned as 250+.

**c.** Number of SuSiE sQTL c.s. per junction, pooled across all tissues and stratified by biotype as in (a).

**d.** Number of variants per sQTL c.s., pooled across all tissues and stratified by biotype as in (a). Counts ≥250 are binned as 250+.

**e.** Fold enrichment of functional annotations among novel lncRNA eQTL (left) and sQTL (right) variants across SuSiE fine-mapping posterior inclusion probability (PIP) thresholds (PIP > 0.01, 0.5, 0.9, 0.95, 0.99), or SuSiE c.s. lead variants. Points, Firth odds ratios (ORs) with allele frequency as a covariate; error bars, 95% CIs; asterisks, two-sided Wald p (* p < 0.05; ** p < 0.01; *** p < 0.001). Variant selection at higher PIP yields higher enrichment effect sizes. SuSiE lead variants do not yield enrichments as high as PIP thresholds.


## Setup


In [ ]:
from pathlib import Path

import json
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 5.0)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

annot_version = "v11"
cs_max_pip_cutoff = -1.0
pip_high = 0.9

biotype_pretty = {
    "novel_lncRNA": "novel lncRNA",
    "existing_lncRNA": "existing lncRNA",
    "protein_coding": "protein-coding",
    "pseudogene": "pseudogene",
    "sRNA": "sRNA",
    "artifact": "artifact",
}
paired = sns.color_palette("Paired", 12)
biotype_palette = {
    "novel_lncRNA": paired[5],
    "existing_lncRNA": paired[4],
    "protein_coding": paired[1],
    "pseudogene": "#734675",
    "sRNA": "#2D716F",
    "artifact": "#cccccc",
}

pip_panels = [0.01, 0.5, 0.9, 0.95, 0.99, cs_max_pip_cutoff]
pip_panel_labels = {
    0.01: "PIP > 0.01",
    0.5: "PIP > 0.5",
    0.9: "PIP > 0.9",
    0.95: "PIP > 0.95",
    0.99: "PIP > 0.99",
    cs_max_pip_cutoff: "lead variant",
}
filter_eqtl = [
    "splicing_merged_lncRNA_d",
    "splicing_merged_protein_coding_d",
    "non_coding_transcript_exon_variant_lncRNA_d",
    "non_coding_transcript_exon_variant_protein_coding_d",
    "promoter_d",
    "intron_variant_lncRNA_d",
    "CTCF_binding_site_d",
    "enhancer_d",
]
pretty_annotations = {
    "splicing_merged_lncRNA_d": "lncRNA\nsplicing",
    "splicing_merged_protein_coding_d": "protein-coding\nsplicing",
    "non_coding_transcript_exon_variant_lncRNA_d": "lncRNA\nexon",
    "non_coding_transcript_exon_variant_protein_coding_d": "protein-coding\nexon",
    "promoter_d": "promoter",
    "intron_variant_lncRNA_d": "lncRNA\nintron",
    "CTCF_binding_site_d": "CTCF",
    "enhancer_d": "enhancer",
}
plot_biotypes = ["novel_lncRNA", "existing_lncRNA", "protein_coding"]


## Paths

In [ ]:
gtex_colors_file = Path("refs/colors.json")
qtl_tissue_ids_file = Path("path/to/qtl_tissue_ids.txt")
gencode_match_file = Path("path/to/gencode_id_matching_lncrna_category.txt")

# Portal SuSiE summaries (one *.SuSiE_summary.parquet per tissue)
eqtl_susie_dir = Path("path/to/v11_eqtl_susie")
sqtl_susie_dir = Path("path/to/v11_sqtl_susie")

# Annotation enrichment (Firth ORs); Zenodo: variant_annotation/
odds_file = Path("path/to/qtl_vep_odds.txt")

fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_9_finemapping_annotations.pdf"
fig_png = fig_dir / "supp_figure_9_finemapping_annotations.png"


## Load data


### QTL tissues and GENCODE IDs


In [ ]:
with open(gtex_colors_file) as f:
    _colors = json.load(f)
    tissue_df = pd.DataFrame.from_dict(_colors, orient="index").reset_index(names="tissue_name")
tissue_df["tissue_file_name"] = (
    tissue_df["tissue_name"]
    .str.replace("(", "", regex=False)
    .str.replace(")", "", regex=False)
    .str.replace(" - ", "_")
    .str.replace(" ", "_")
)
qtl_tissues = set(pd.read_table(qtl_tissue_ids_file, header=None)[0])
tissue_df = tissue_df[tissue_df["tissue_file_name"].isin(qtl_tissues)].copy()
gencode_ids = pd.read_table(gencode_match_file, low_memory=False)
gencode_ids = gencode_ids.rename(columns={"gene_type_grouped_highlight": "gene_type_grouped"})
print(f"QTL tissues: {len(tissue_df):,}; GENCODE IDs: {len(gencode_ids):,}")


### SuSiE credible sets


In [ ]:
def load_susie_df(tissue_df, gencode_ids, results_dir, file_end, desc):
    frames, tissue_ids = [], []
    for _, row in tqdm(tissue_df.iterrows(), total=len(tissue_df), desc=desc):
        path = results_dir / f"{row['tissue_file_name']}.{file_end}.SuSiE_summary.parquet"
        tissue_cs = pd.read_parquet(
            path, columns=["phenotype_id", "cs_id", "cs_size", "variant_id", "pip"]
        )
        # One row per credible set; for single-variant sets this row is the sole variant
        frames.append(tissue_cs.drop_duplicates(["phenotype_id", "cs_id"]))
        tissue_ids.append(row["tissue_file_name"])

    df = pd.concat(frames, keys=tissue_ids, names=["tissue_id", "idx"])
    df = df.reset_index(level="idx", drop=True).reset_index()
    df["gene_id_stable"] = df["phenotype_id"].str.split(".").str[0].str.split(":").str[-1]
    df = df.merge(
        gencode_ids[["gene_id_stable", "gene_type_grouped"]],
        on="gene_id_stable",
        how="left",
    )
    df["biotype"] = df["gene_type_grouped"]
    return df


def summarize_credible_sets(df, qtl_label):
    cs = df.drop_duplicates(subset=["tissue_id", "phenotype_id", "cs_id"]).copy()
    per_tissue_pheno = (
        cs.groupby(["tissue_id", "phenotype_id", "biotype"])
        .size()
        .rename("n_cs")
        .reset_index()
    )
    per_tissue_pheno["qtl_type"] = qtl_label
    cs["qtl_type"] = qtl_label
    return per_tissue_pheno, cs


eqtl_df = load_susie_df(tissue_df, gencode_ids, eqtl_susie_dir, "v11.eQTLs", "eQTL")
sqtl_df = load_susie_df(tissue_df, gencode_ids, sqtl_susie_dir, "v11.sQTLs", "sQTL")
per_tissue_pheno_e, per_cs_e = summarize_credible_sets(eqtl_df, "eQTL")
per_tissue_pheno_s, per_cs_s = summarize_credible_sets(sqtl_df, "sQTL")
per_tissue_pheno_all = pd.concat([per_tissue_pheno_e, per_tissue_pheno_s], ignore_index=True)
per_cs_all = pd.concat([per_cs_e, per_cs_s], ignore_index=True)
print(f"tissue×phenotype rows: {len(per_tissue_pheno_all):,}; credible sets: {len(per_cs_all):,}")


### Annotation odds (PIP thresholds)


In [ ]:
odds = pd.read_table(odds_file)
odds = odds[odds["gene_type"] == "novel_lncRNA"].copy()
odds["annotation"] = odds["annotation"].str.split("_v1").str[0]
available_pips = sorted(odds["pip_cutoff"].dropna().unique())
print(f"Loaded novel lncRNA odds ({annot_version}); pip_cutoff values: {available_pips}")
missing = [p for p in pip_panels if p not in available_pips]
if missing:
    raise ValueError(f"Missing pip_cutoff(s): {missing}")


## Run analysis


### PIP-panel annotation order


In [ ]:
def get_panel_data(odds_df, susie_type, pip, gene_type, filter_annots=None):
    sub = odds_df[
        (odds_df["pip_cutoff"] == pip)
        & (odds_df["annot_version"] == annot_version)
        & (odds_df["susie_type"] == susie_type)
        & (odds_df["gene_type"] == gene_type)
    ].copy()
    sub = sub.dropna(subset=["odds_ratio", "ci_lower", "ci_upper"], how="all")
    if filter_annots is not None:
        sub = sub[sub["annotation"].isin(filter_annots)]
    return sub


orders = []
for susie_type in ("eqtl", "sqtl"):
    sub = get_panel_data(odds, susie_type, 0.01, "novel_lncRNA", filter_annots=filter_eqtl)
    order = sub.groupby("annotation")["odds_ratio"].max().sort_values(ascending=False).index.tolist()
    orders.append(order)
annot_order = max(orders, key=len) if orders else []
if not annot_order:
    annot_order = [a for a in filter_eqtl if a in odds["annotation"].unique()]
print("annotation order:", [pretty_annotations.get(a, a) for a in annot_order])


### Novel lncRNA credible-set summaries (panels a–d)


In [ ]:
# Per QTL type: % of novel lncRNA phenotypes (tissue × phenotype) with >1 credible set,
# % of credible sets that are single-variant with PIP >= pip_high, and unique variants in those sets
susie_stats = {}
for qtl_label, per_pheno, per_cs in (
    ("eQTL", per_tissue_pheno_e, per_cs_e),
    ("sQTL", per_tissue_pheno_s, per_cs_s),
):
    pheno = per_pheno[per_pheno["biotype"] == "novel_lncRNA"]
    cs = per_cs[per_cs["biotype"] == "novel_lncRNA"]
    single_high = cs[(cs["cs_size"] == 1) & (cs["pip"] >= pip_high)]
    susie_stats[qtl_label] = {
        "multi_cs_pct": 100 * (pheno["n_cs"] > 1).mean(),
        "single_high_pip_pct": 100 * len(single_high) / len(cs),
        "high_conf_variants": single_high["variant_id"].nunique(),
    }


## Make figure


In [ ]:
biotypes = [b for b in plot_biotypes if b in set(per_tissue_pheno_all["biotype"])]
n_measures = 4

fig = plt.figure(figsize=(7.2, 5.0), dpi=300)
outer = fig.add_gridspec(2, 1, height_ratios=[0.45, 1.55], hspace=0.58, top=0.90)
gs = outer[0].subgridspec(1, n_measures, wspace=0.50)
axes_hist = [fig.add_subplot(gs[0, j]) for j in range(n_measures)]

cap = int(
    per_tissue_pheno_all.loc[per_tissue_pheno_all["biotype"].isin(biotypes), "n_cs"].max()
)
cs_bins = np.arange(0.5, cap + 1.5, 1)
cs_xticks = list(range(1, cap + 1))
cs_xlim = (0.5, cap + .5)

size_cap = 250
size_bins = np.arange(0.5, size_cap + 1.5, 1)
size_xticks = [1, 50, 100, 150, 200, 250]
size_xticklabels = ["1", "50", "100", "150", "200", "250+"]
size_xlim = (0.5, size_cap + 1.5)

hist_specs = [
    ("eQTL", "n_cs", "eQTL c.s. / gene", "genes", False),
    ("eQTL", "cs_size", "eQTL variants / c.s.", "genes", True),
    ("sQTL", "n_cs", "sQTL c.s. / junction", "junctions", False),
    ("sQTL", "cs_size", "sQTL variants / c.s.", "junctions", True),
]
for ax, (qtl_label, value_col, xlabel, ylabel, is_size) in zip(axes_hist, hist_specs):
    src = per_cs_all if is_size else per_tissue_pheno_all
    bins = size_bins if is_size else cs_bins
    for bt in biotypes:
        sub = src.loc[(src["qtl_type"] == qtl_label) & (src["biotype"] == bt)]
        vals = sub[value_col].to_numpy()
        if is_size:
            vals = np.clip(vals, None, size_cap)
        color = biotype_palette.get(bt, "#888888")
        ax.hist(
            vals, bins=bins, histtype="step", color=color, linewidth=0.8,
            label=biotype_pretty[bt],
        )
    ax.set_yscale("log")
    ax.set_ylim(bottom=0.7)
    if is_size:
        ax.set_xlim(*size_xlim)
        ax.set_xticks(size_xticks)
        ax.set_xticklabels(size_xticklabels)
    else:
        ax.set_xlim(*cs_xlim)
        ax.set_xticks(cs_xticks)
    ax.set_ylabel(ylabel)
    ax.set_xlabel(xlabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    biotype_handles = [
        Line2D([0], [0], color=biotype_palette[bt], lw=0.8, label=biotype_pretty[bt])
        for bt in biotypes
    ]
    ax.legend(
        handles=biotype_handles, loc="upper right", frameon=False,
        fontsize=4, handlelength=1.2,
    )

gs_e = outer[1].subgridspec(1, 2, wspace=0.18)
gene_type = "novel_lncRNA"
qtl_cols = [("eqtl", "eVars"), ("sqtl", "sVars")]
pip_palette = dict(zip(pip_panels, sns.color_palette("viridis", len(pip_panels))))
pip_offsets = dict(zip(pip_panels, np.linspace(-0.12, 0.12, len(pip_panels))))

n_ann = len(annot_order)
ymap = {ann: i for i, ann in enumerate(annot_order)}
ann_labels = [pretty_annotations.get(ann, ann) for ann in annot_order]

axes_e = [fig.add_subplot(gs_e[0, j]) for j in range(2)]
for col, (susie_type, var_label) in enumerate(qtl_cols):
    ax = axes_e[col]
    for i in range(n_ann):
        if i % 2 == 1:
            ax.axhspan(i - 0.5, i + 0.5, color="#eeeeee", zorder=0, clip_on=True)
    ax.axvline(0, color="k", linestyle=":", linewidth=1, zorder=2)
    for pip in pip_panels:
        color = pip_palette[pip]
        d = get_panel_data(odds, susie_type, pip, gene_type, filter_annots=filter_eqtl)
        d = d[d["annotation"].isin(annot_order)]
        if d.empty:
            continue
        y = d["annotation"].map(ymap).to_numpy() + pip_offsets[pip]
        x = np.log2(d["odds_ratio"].to_numpy())
        xerr_lo = x - np.log2(d["ci_lower"].to_numpy())
        xerr_hi = np.log2(d["ci_upper"].to_numpy()) - x
        ax.errorbar(
            x=x, y=y, xerr=[xerr_lo, xerr_hi], color=color, marker="o",
            markerfacecolor=color, markeredgecolor="k", markeredgewidth=0.35,
            ecolor=color,
            capsize=0, markersize=2, linestyle="none", zorder=2, alpha=0.9, elinewidth=0.8,
        )
        pvals = d["p_value"].to_numpy() if "p_value" in d.columns else np.full(len(d), np.nan)
        for xi, yi, p in zip(x, y, pvals):
            if not np.isfinite(p):
                stars = ""
            elif p < 0.001:
                stars = "***"
            elif p < 0.01:
                stars = "**"
            elif p < 0.05:
                stars = "*"
            else:
                stars = ""
            if stars and np.isfinite(xi):
                ax.text(xi, yi - 0.10, stars, va="center", ha="right", fontsize=3.5, color=color)
    ax.set_xlabel(f"Fold enrichment in novel lncRNA {var_label}")
    ax.set_ylim(n_ann - 0.5, -0.5)
    ax.set_xticks([-2, -1, 0, 1, 2, 3, 4, 5, 6, 7])
    ax.set_xticklabels(["1/4", "1/2", 1, 2, 4, 8, 16, 32, 64, 128])
    ax.set_xlim(-2, 7)
    if col == 0:
        ax.set_yticks(range(n_ann))
        ax.set_yticklabels(ann_labels)
        ax.tick_params(axis="y", length=0, pad=2)
    else:
        ax.set_yticks([])
    sns.despine(ax=ax, left=True)

legend_handles = [
    Line2D(
        [0], [0], marker="o", color="w", markerfacecolor=pip_palette[pip],
        markeredgecolor="k", markeredgewidth=0.35, markersize=3, linestyle="None",
        label=pip_panel_labels[pip],
    )
    for pip in pip_panels
]
axes_e[0].legend(
    handles=legend_handles, loc="upper center", ncol=len(legend_handles),
    bbox_to_anchor=(1.0, 1.08), frameon=False, fontsize=5,
)

_inv = fig.transFigure.inverted()
_x_left = min(
    _inv.transform(ax.transAxes.transform((-0.08, 0)))[0]
    for ax in (axes_hist[0], axes_e[0])
)
for ax, label in zip(axes_hist, "abcd"):
    _x = _x_left if label == "a" else _inv.transform(ax.transAxes.transform((-0.15, 0)))[0]
    _y = _inv.transform(ax.transAxes.transform((0, 1.12)))[1]
    fig.text(_x, _y, label, fontweight="bold", va="bottom", ha="right", fontsize=9)
_y_e = _inv.transform(axes_e[0].transAxes.transform((0, 1.03)))[1]
fig.text(_x_left, _y_e, "e", fontweight="bold", va="bottom", ha="right", fontsize=9)

plt.show()


## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")


## Print stats


In [ ]:
# Results (novel lncRNAs): phenotypes with >1 credible set; single-variant credible sets with PIP >= 0.9; unique variants
for qtl_label, phenotype_label in (("eQTL", "expression"), ("sQTL", "splicing")):
    stats = susie_stats[qtl_label]
    print(f"{qtl_label}: {stats['multi_cs_pct']:.0f}% of novel lncRNA {phenotype_label} phenotypes have >1 credible set")
    print(f"{qtl_label}: {stats['single_high_pip_pct']:.0f}% of credible sets are single-variant with PIP >= {pip_high}")
    print(f"{qtl_label}: {stats['high_conf_variants']:,} unique high-confidence variants across tissues")
